# Med-Drishti: BioBERT Clinical Triage & Red-Flag Classifier on Kaggle GPU

This notebook fine-tunes **BioBERT** (`dmis-lab/biobert-v1.1`) or **Bio_ClinicalBERT** (`emilyalsentzer/Bio_ClinicalBERT`) on patient intake text to classify severity levels (`Emergency`, `Urgent`, `Routine`) and detect clinical red flags.

### Kaggle Setup Instructions:
1. Accelerator: Select **GPU T4 x2** or **GPU P100** under Notebook Settings.
2. Internet: Turn **ON** to download HuggingFace models and datasets.

In [ ]:
# Step 1: Install necessary libraries
import sys
import subprocess

packages = ['transformers', 'datasets', 'evaluate', 'accelerate', 'scikit-learn', 'torch', 'pandas']
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + packages)

In [ ]:
import os
import torch
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from transformers import (
    AutoTokenizer, 
    AutoModelForSequenceClassification, 
    Trainer, 
    TrainingArguments
)
from datasets import Dataset

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[Med-Drishti ML] Using device: {device}")

ModuleNotFoundError: No module named 'torch'

In [ ]:
# Step 2: Define Triage Categories & Sample Synthetic Dataset Generator
LABEL_MAP = {
    0: "ROUTINE",
    1: "URGENT",
    2: "EMERGENCY_RED_FLAG"
}

def load_sample_clinical_data():
    data = [
        {"text": "Patient reports mild runny nose and cough for 2 days. No fever.", "label": 0},
        {"text": "Severe crushing chest pain radiating to left arm with diaphoresis and shortness of breath.", "label": 2},
        {"text": "High fever 103F for 4 days with chills and body pain.", "label": 1},
        {"text": "Sudden onset right-sided facial drooping, arm weakness, and slurred speech.", "label": 2},
        {"text": "Routine checkup for blood pressure medication refill.", "label": 0},
        {"text": "Abdominal pain in lower right quadrant with vomiting and fever.", "label": 1},
        {"text": "Stridor, severe allergic reaction, swelling of lips and throat.", "label": 2},
        {"text": "Mild knee pain after walking for long hours.", "label": 0}
    ]
    return pd.DataFrame(data)

df = load_sample_clinical_data()
print(f"Loaded dataset with {len(df)} samples.")

In [ ]:
# Step 3: Load BioBERT Tokenizer & Model
MODEL_NAME = "emilyalsentzer/Bio_ClinicalBERT"
print(f"Loading clinical pretrained model: {MODEL_NAME}")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, 
    num_labels=len(LABEL_MAP),
    id2label=LABEL_MAP,
    label2id={v: k for k, v in LABEL_MAP.items()}
)
model.to(device)

In [ ]:
# Step 4: Tokenize Dataset
def tokenize_function(examples):
    return tokenizer(examples["text"], padding="max_length", truncation=True, max_length=128)

raw_dataset = Dataset.from_pandas(df)
tokenized_dataset = raw_dataset.map(tokenize_function, batched=True)
train_test_split_ds = tokenized_dataset.train_test_split(test_size=0.2)

train_ds = train_test_split_ds["train"]
eval_ds = train_test_split_ds["test"]

In [ ]:
# Step 5: Training & Evaluation Setup
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    macro_f1 = f1_score(labels, predictions, average="macro")
    return {"macro_f1": macro_f1}

OUTPUT_DIR = "/kaggle/working/med_drishti_triage_model"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    fp16=torch.cuda.is_available(),
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer initialized ready for model fine-tuning.")

In [ ]:
# Step 6: Save Trained Model for Backend Loading
def export_triage_model(model, tokenizer, path):
    os.makedirs(path, exist_ok=True)
    model.save_pretrained(path)
    tokenizer.save_pretrained(path)
    print(f"✓ Fine-tuned Clinical Triage model successfully exported to: {path}")

# Call when training finishes:
# export_triage_model(model, tokenizer, "/kaggle/working/med_drishti_triage_final")